In [1]:
import mdtraj as md
import os
import glob
import pandas as pd

In [2]:
output_path='/global/cfs/cdirs/m4704/100125_Nature_Com_data/single_conformation/nma'

In [3]:
ref_path='/global/cfs/cdirs/m4704/100125_Nature_Com_data/Apo_holo_data/pdbs'

In [4]:
def generate_rmsd_alphaSAXS(pdb_id,ref_path,output_path):
    rmsd=[]
    ref_t=md.load(os.path.join(ref_path,pdb_id+'.pdb'))
    ref_atom=ref_t.topology.select('name CA')
    test_t = md.load(os.path.join(output_path, pdb_id+'.pdb'))
    test_atom=test_t.topology.select('name CA')
    if len(test_atom)!=len(ref_atom):
        return pdb_id+' has the wrong length'
    ca_selection=md.rmsd(test_t,ref_t,0,test_atom,ref_atom)
    return ca_selection[0]*10

In [5]:
pdb_list=[ f[:-4] for f in os.listdir(ref_path) if f.endswith('.pdb')]

In [6]:
len(pdb_list)

80

In [7]:
result={}

In [8]:
for i in pdb_list:
    try:
        result[i]=generate_rmsd_alphaSAXS(i,ref_path,output_path)
    except:
        print(f'{i} is not in the prediction dataset')

In [9]:
result

{'2K8R-1_A': 4.274497926235199,
 '1XSA-23_A': 4.290747940540314,
 '1K2H-4_A': 6.59506618976593,
 '2D9E-12_A': 3.833523690700531,
 '1P7M-18_A': 2.273441255092621,
 '1EX6_B': 5.6531500816345215,
 '2AI6-18_A': 2.9643553495407104,
 '1MO8-14_A': 20.744996070861816,
 '2KXL-8_A': 4.462212920188904,
 '1TJD_A': 1.7837749421596527,
 '2VCD-8_A': 1.5808682143688202,
 '2F63-4_A': 2.054574489593506,
 '1ROE-10_A': 4.488076269626617,
 '1CZ2-8_A': 2.7978739142417908,
 '1PUN-7_A': 3.803841769695282,
 '1WCW_A': 2.3838433623313904,
 '1ZFS-13_B': 2.433135360479355,
 '1VHL_A': 2.6123732328414917,
 '2LKC-4_A': 8.724854588508606,
 '1ZOL_A': 3.0302557349205017,
 '1NTR-7_A': 3.999658226966858,
 '1UR6-4_A': 2.1775028109550476,
 '1TNQ-33_A': 6.5153968334198,
 '1S2O_A': 3.007400929927826,
 '1JM4-15_B': 5.122198462486267,
 '1RRO_A': 0.8389937877655029,
 '2CG6_A': 6.507254242897034,
 '1MO7-3_A': 21.899242401123047,
 '1VIY_C': 2.646643817424774,
 '1RTP_1': 0.6280387938022614,
 '1JKN-15_A': 3.087535798549652,
 '1JFJ-3

In [10]:
df = pd.DataFrame(result.items(), columns=['Key', 'Value'])

In [11]:
df.describe()

,Value
count,80.000000
mean,4.702575
std,4.567831
min,0.420289
25%,2.270763
50%,3.099344
75%,4.838864
max,21.899242


In [12]:
pair_csv=pd.read_csv('/global/cfs/cdirs/m4704/100125_Nature_Com_data/Apo_holo_data/Table_rmsd_Apo_vs_Holo.csv',sep=';')

In [13]:
pair_dict={}
for index, i in pair_csv.iterrows():
    pair_dict[i['Apo_ID']]=i['Holo_ID']
    #pair_dict[i['Holo_ID']]=i['Apo_ID']

In [14]:
result_csv=pd.DataFrame.from_dict(result,orient='index').reset_index().rename(columns={'index':'ID',0:'RMSD'})

In [15]:
result_csv['pair_ID']=result_csv['ID'].map(pair_dict)

In [16]:
result_pair=result_csv.dropna().merge(result_csv[['ID',
'RMSD']],left_on='pair_ID',right_on='ID',suffixes=['','_pair']).drop(columns={'pair_ID'})

In [17]:
result_pair

,ID,RMSD,ID_pair,RMSD_pair
0,1XSA-23_A,4.290748,1XSC-20_A,2.667385
1,1K2H-4_A,6.595066,1ZFS-13_B,2.433135
2,2D9E-12_A,3.833524,2RS9-13_B,3.111152
3,1EX6_B,5.653150,1EX7_A,1.731599
4,2AI6-18_A,2.964355,2OZW-3_A,3.832838
5,2KXL-8_A,4.462213,2K0G-9_A,3.537801
6,1TJD_A,1.783775,1EEJ_B,2.574440
7,2F63-4_A,2.054574,1EQM_A,4.000775
8,2LKC-4_A,8.724855,2LKD-18_A,5.908632
9,1ZOL_A,3.030256,1O03_A,0.955561


In [19]:
AlphaFold_df=pd.read_csv('AlphaFold_RMSD.csv',index_col='ID')
AlphaFold_df=AlphaFold_df.drop(AlphaFold_df.columns[0], axis=1)

In [20]:
AlphaFold_df['RMSD_Fold'] = pd.to_numeric(AlphaFold_df['RMSD_Fold'], errors='coerce')

In [21]:
AlphaFold_df=AlphaFold_df.dropna()

In [22]:
AlphaFold_dict=AlphaFold_df.to_dict()['RMSD_Fold']

In [23]:
result_pair.dtypes

ID            object
RMSD         float64
ID_pair       object
RMSD_pair    float64
dtype: object

In [24]:
AlphaFold_df.dtypes

RMSD_Fold    float64
dtype: object

In [25]:
for index, i in result_pair.iterrows():
    result_pair.loc[index,'RMSD']-=AlphaFold_dict[i['ID']]
    result_pair.loc[index,'RMSD_pair']-=AlphaFold_dict[i['ID_pair']]

In [27]:
result_pair['RMSD_sum']=result_pair['RMSD']+result_pair['RMSD_pair']

In [28]:
result_pair.sort_values('RMSD_sum')

,ID,RMSD,ID_pair,RMSD_pair,RMSD_sum
28,4AKE_B,-3.357762,2ECK_B,-3.759513,-7.117275
25,1FMF-4_A,-1.511834,1ID8-11_A,-1.478940,-2.990774
6,1TJD_A,-0.712956,1EEJ_B,-0.441966,-1.154923
27,2L50-24_A,-0.429687,2L51-15_B,-0.681335,-1.111022
10,1NTR-7_A,-0.378808,1KRX-4_A,-0.487784,-0.866592
15,1JFJ-3_A,-0.071992,1JFK_A,-0.402335,-0.474327
16,1LIP-2_A,-0.495849,1JTB-6_A,0.065737,-0.430112
18,2RCS_H,1.211920,1AJ7_H,-1.491750,-0.279830
39,1MUT-11_A,-0.051301,1PUN-7_A,-0.226884,-0.278185
22,2NLN-4_A,-0.414035,1RRO_A,0.223240,-0.190795
